In [1]:
from pydash import filter_

from lib.sources import load_faq_documents
from lib.types import FAQDocument, FAQGroundTruthRecord


predicate = {"course": "llm-zoomcamp"}

documents: list[FAQDocument] = filter_(load_faq_documents(), predicate)

In [2]:
print(documents[0]["id"])
print(documents[0]["course"])
print(documents[0]["section"])
print(documents[0]["question"])
print(documents[0]["answer"])

74eb249bbf
llm-zoomcamp
General Course-Related Questions
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [3]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [4]:
how_to_generate_questions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [5]:
from dotenv import dotenv_values
from openai import OpenAI


config = dotenv_values()

openai_client = OpenAI(api_key=config["OPENAI_API_KEY"])

In [6]:
import json


user_prompt = json.dumps(documents[0])

In [7]:
from openai.types.responses import ResponseInputParam


messages: ResponseInputParam = [
    {"role": "developer", "content": how_to_generate_questions},
    {"role": "user", "content": user_prompt}
]

In [8]:
response = openai_client.responses.parse(
    model="gpt-4.1-mini",
    input=messages,
    text_format=Questions,
)

In [9]:
direct_result = response.output_parsed

assert direct_result is not None

print(direct_result)

questions=['Can I join the LLM Zoomcamp course even if I just found out about it?', 'Is it still possible to get a certificate if I join the course now?', 'What do I need to do to get a certificate from the course?', 'Are there deadlines for submitting the project to get a certificate?', 'If I start late, can I still complete the course and get recognized for it?']


In [10]:
from lib.llm import call_structured_llm


call = call_structured_llm(
    openai_client,
    instructions=how_to_generate_questions,
    user_prompt=user_prompt,
    output_type=Questions
)

print("Questions:")
for question in call.result.questions:
    print(question)


print(
    f"Cost - Input: {call.metrics.input_tokens}, "
    f"Output: {call.metrics.output_tokens}"
)

Questions:
I just found this course — is it still possible to join now?
Am I allowed to start the course late, or did I miss my chance?
If I join after the course has already started, can I still get a certificate?
What do I need to do to be eligible for the certificate if I sign up now?
Is there still time to submit the project for certification, or is the deadline already over?
Cost - Input: 207, Output: 99


In [11]:
call.metrics.price

UsagePrice(input_cost_usd=0.00015525, output_cost_usd=0.0004455)

In [12]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [13]:
doc = documents[0]

records: list[FAQGroundTruthRecord] = []

for q in direct_result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

print(records)

[{'question': 'Can I join the LLM Zoomcamp course even if I just found out about it?', 'document': '74eb249bbf'}, {'question': 'Is it still possible to get a certificate if I join the course now?', 'document': '74eb249bbf'}, {'question': 'What do I need to do to get a certificate from the course?', 'document': '74eb249bbf'}, {'question': 'Are there deadlines for submitting the project to get a certificate?', 'document': '74eb249bbf'}, {'question': 'If I start late, can I still complete the course and get recognized for it?', 'document': '74eb249bbf'}]


In [14]:
from evaluation_utils import call_with_retry
from lib.llm import call_structured_llm
from lib.metrics import ModelCallMetrics


def generate_ground_truth(
    doc: FAQDocument,
) -> tuple[list[FAQGroundTruthRecord], ModelCallMetrics]:

    user_prompt = json.dumps(doc)

    call = call_with_retry(
        lambda: call_structured_llm(
            openai_client,
            instructions=how_to_generate_questions,
            user_prompt=user_prompt,
            output_type=Questions,
        )
    )

    results: list[FAQGroundTruthRecord] = []

    for q in call.result.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, call.metrics

In [15]:
from tqdm.auto import tqdm


ground_truth: list[FAQGroundTruthRecord] = []
call_metrics: list[ModelCallMetrics] = []

for doc in tqdm(documents[:5]):
    generated_records, metrics = generate_ground_truth(doc)
    ground_truth.extend(generated_records)
    call_metrics.append(metrics)

  0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
for _ in ground_truth:
    print(_)

{'question': 'Can I still join the course if I only found it now?', 'document': '74eb249bbf'}
{'question': 'Is it too late to start the course, or can I jump in later?', 'document': '74eb249bbf'}
{'question': 'If I join after the course already started, can I still get a certificate?', 'document': '74eb249bbf'}
{'question': 'Do I need to submit the project before submissions close in order to get the certificate?', 'document': '74eb249bbf'}
{'question': 'What happens if I start the course now but miss the project submission deadline?', 'document': '74eb249bbf'}
{'question': 'I registered for the LLM Zoomcamp — when should I expect the confirmation email, or am I supposed to get one at all?', 'document': '977bf7786c'}
{'question': 'Do I need to wait for approval after signing up for the course, or can I just start right away?', 'document': '977bf7786c'}
{'question': 'Is there some kind of accepted list for the LLM Zoomcamp registration, or is signing up enough?', 'document': '977bf7786c

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress


with ThreadPoolExecutor(max_workers=6) as pool:
    results: list[tuple[list[FAQGroundTruthRecord], ModelCallMetrics]] = (
        map_progress(pool, documents, generate_ground_truth))

  0%|          | 0/113 [00:00<?, ?it/s]

In [18]:
ground_truth: list[FAQGroundTruthRecord] = []
call_metrics: list[ModelCallMetrics] = []

for records, metrics in results:
    ground_truth.extend(records)
    call_metrics.append(metrics)

len(ground_truth)

565

In [19]:
sum(metrics.price.total_cost_usd for metrics in call_metrics)

0.0874455

In [20]:
import pandas as pd


ground_truth_df = pd.DataFrame(ground_truth)

In [21]:
ground_truth_df.to_csv("data/faq_ground_truth.csv", index=False)